In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
import torch
import numpy as np
import cv2
from PIL import Image
import warnings
warnings.filterwarnings("ignore")

print("✅ Imports done")
print("CUDA:", torch.cuda.is_available())

✅ Imports done
CUDA: True


In [3]:
CONFIG = {
    "DEVICE": "0",
    "NAME": "id0_train",
    "TOKEN_NUM": 8,                 # reduced for memory
    "MAX_IMAGE_SIZE": 224,          # reduced for memory
    "PRETRAIN_PATH": "./checkpoints/checkpoints_attr_Stage2_merge",
    "VIP_PATH": "./checkpoints/Stage3/id0_train/vip_token.pt",
    "FACE_CENTER": "./FaceDATA/FaceEmb_Center/id0.ckpt"
}

In [4]:
def load_models():
    print("🚀 Loading models...")

    from Models.VIPGuard import Qwen2_5_VLForConditionalGeneration, Qwen2_5_VLProcessor
    from Models.Wrapper import Wrapper
    from Models.Face_Model.FaceModel import FG_Face

    torch.cuda.empty_cache()

    vl_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    CONFIG["PRETRAIN_PATH"],
    device_map="balanced",   # ✅ key fix
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
    )

    processor = Qwen2_5_VLProcessor.from_pretrained(CONFIG["PRETRAIN_PATH"])

    model = Wrapper(
        vl_model=vl_model,
        processor=processor,
        optim_facechecker=False,
        vip_token_num=CONFIG["TOKEN_NUM"]
    )

    # ❌ DO NOT move model to cuda

    face_model = FG_Face(
        attributes='',
        token_dim=3584,
        model_name='transface'
    )
    face_model.to("cpu")

    return model, processor, face_model

In [5]:
# ============================
# 4. LOAD VIP TOKEN
# ============================
def load_vip(model):
    print("📥 Loading VIP token...")

    state = torch.load(CONFIG["VIP_PATH"], map_location="cpu")
    model.load_vip(state)

    print("✅ VIP token loaded")
    return model

In [6]:
# ============================
# 5. LOAD FACE CENTER
# ============================
def load_face_center():
    print("📥 Loading face center...")

    center = torch.load(CONFIG["FACE_CENTER"], map_location="cpu")

    print("✅ Face center loaded")
    return center


In [7]:

# ============================
# 6. FACE SIMILARITY
# ============================
def preprocess(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (112, 112))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.transpose(2, 0, 1)

    img = torch.tensor(img).unsqueeze(0).float()
    img = img / 255.0
    img = (img - 0.5) / 0.5

    return img


def get_similarity(img_path, face_model, center):
    with torch.no_grad():
        img = preprocess(img_path)
        emb = face_model.face_emb_forward(img).squeeze()

        emb = torch.nn.functional.normalize(emb, dim=0)
        center = torch.nn.functional.normalize(center, dim=0)

        sim = torch.cosine_similarity(emb, center, dim=0)
        sim = 0.5 + 0.5 * sim.item()

        return int(sim * 100)

In [25]:
# ============================
# 7. INFERENCE
# ============================
def infer(img_path, model, processor, face_model, center):
    from qwen_vl_utils import process_vision_info

    print(f"\n🖼️ Testing: {img_path}")

    sim = get_similarity(img_path, face_model, center)
    print("Similarity:", sim)

    message = [{
        "role": "user",
        "content": [
            {"type": "text", "text": "<|face_pad|>"},
            {"type": "image", "image": img_path},
            {"type": "text",
             "text": f"Similarity {sim}/100. Is this VIP person? Answer yes or no."}
        ]
    }]

    text = processor.apply_chat_template(message, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(message)

    # resize (memory safe)
    for i in range(len(image_inputs)):
        h, w = image_inputs[i].size
        if max(h, w) > CONFIG["MAX_IMAGE_SIZE"]:
            scale = CONFIG["MAX_IMAGE_SIZE"] / max(h, w)
            image_inputs[i] = image_inputs[i].resize(
                (int(w*scale), int(h*scale))
            )

    inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    return_tensors="pt",
    padding=True,

    our_token=True,
    our_token_length=CONFIG["TOKEN_NUM"],   # ✅ FIX
    face_pad=True,
    face_length=model.vl_model.facechecker.face_checker.vip_prompt.data.shape[0]
)

    # move only tensors
    inputs = {
    k: v.to("cuda").to(torch.float16) if v.dtype == torch.float32 else v.to("cuda")
    for k, v in inputs.items()
}
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=100)

    out = processor.batch_decode(output, skip_special_tokens=True)[0]

    print("\n🧠 Model Output:")
    print(out)


In [26]:
import torch
torch.cuda.empty_cache()

In [27]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [28]:
# ============================
# 8. RUN PIPELINE
# ============================

# 1️⃣ Load models
model, processor, face_model = load_models()

print("📥 Loading VIP token manually...")

state = torch.load(CONFIG["VIP_PATH"], map_location="cpu")

print("DEBUG type:", type(state))

if isinstance(state, dict):
    print("📦 Detected dict format")

    if 'vip_token' in state:
        print("➡️ Loading vip_token as tensor")
        model.vl_model.facechecker.face_checker.vip_prompt.data = state['vip_token']

    elif 'facechecker' in state:
        print("➡️ Using full facechecker state_dict")
        model.vl_model.facechecker.load_state_dict(state['facechecker'])

    else:
        print("⚠️ Unknown dict structure:", state.keys())

elif isinstance(state, torch.nn.Parameter) or isinstance(state, torch.Tensor):
    print("📦 Detected tensor format")
    model.vl_model.facechecker.face_checker.vip_prompt.data = state

else:
    print("❌ Unsupported format:", type(state))

print("✅ VIP Token Loaded Successfully!")

model = model.half()

# 3️⃣ Load face center
center = load_face_center()

# 4️⃣ Run inference
infer(
    "./FaceDATA/Training_Img/id0/r_efs_i/id0_1/fake.png",
    model,
    processor,
    face_model,
    center
)


🚀 Loading models...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


Loading checkpoint shards: 100%|██████████| 4/4 [00:39<00:00,  9.92s/it]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


Load Face Model Successfully, The Model is TransFace
📥 Loading VIP token manually...
DEBUG type: <class 'dict'>
📦 Detected dict format
➡️ Loading vip_token as tensor
✅ VIP Token Loaded Successfully!
📥 Loading face center...
✅ Face center loaded

🖼️ Testing: ./FaceDATA/Training_Img/id0/r_efs_i/id0_1/fake.png


Keyword argument `our_token` is not a valid argument for this processor and will be ignored.
Keyword argument `our_token_length` is not a valid argument for this processor and will be ignored.
Keyword argument `face_pad` is not a valid argument for this processor and will be ignored.
Keyword argument `face_length` is not a valid argument for this processor and will be ignored.


Similarity: 79

🧠 Model Output:
system
You are a helpful assistant.
user
Similarity 79/100. Is this VIP person? Answer yes or no.
assistant
<Conclusion>[No] The individual is not a verified VIP.</Conclusion>



In [31]:
infer(
    "./FaceDATA/Training_Img/id0/r_efs_i/id0_1/fake.png",
    model,
    processor,
    face_model,
    center
)


🖼️ Testing: ./FaceDATA/Training_Img/id0/r_efs_i/id0_1/fake.png
Similarity: 79

🧠 Model Output:
system
You are a helpful assistant.
user
Similarity 79/100. Is this VIP person? Answer yes or no.
assistant
<Conclusion>[No] The individual is not a verified VIP.</Conclusion>

